# 01 — Exploratory Data Analysis: Amazon Reviews

## ¿Qué es este dataset?

El dataset **Amazon Reviews** contiene reseñas de productos de Amazon con las siguientes columnas clave:

| Columna | Descripción |
|---------|-------------|
| `Id` | Identificador único de la reseña |
| `ProductId` | Identificador del producto reseñado |
| `UserId` | Identificador del usuario |
| `ProfileName` | Nombre del perfil del usuario |
| `HelpfulnessNumerator` | Número de usuarios que encontraron útil la reseña |
| `HelpfulnessDenominator` | Total de usuarios que evaluaron la utilidad |
| `Score` | Calificación en estrellas (1–5) |
| `Time` | Timestamp de la reseña |
| `Summary` | Título/resumen de la reseña |
| `Text` | Texto completo de la reseña |

## ¿Por qué este EDA?

Antes de construir el pipeline medallion (Bronze → Silver → Gold) y entrenar el clasificador de urgencia, necesitamos entender:

1. **Volumen y completitud** — ¿Cuántas reseñas hay? ¿Hay nulos?
2. **Distribución de estrellas** — ¿Está balanceado o hay sesgo?
3. **Distribución de urgencia** — ¿Cómo se mapean las estrellas a niveles de urgencia?
4. **Longitud del texto** — ¿Qué tan largas son las reseñas? Esto impacta el chunking y los embeddings.
5. **Ejemplos por clase** — ¿Qué textos tiene cada nivel de urgencia?

El objetivo es que la clase **negativa** (Score 1-2) tenga más peso en urgencia, ya que son las reseñas que requieren atención inmediata.

## Setup

Cargamos las librerías necesarias y leemos el dataset desde MinIO (bucket `bronze/raw/Reviews.csv`).

In [ ]:
import io
import os

import boto3
import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

# MinIO client
client = boto3.client(
    "s3",
    endpoint_url=os.getenv("MINIO_ENDPOINT", "http://minio:9000"),
    aws_access_key_id=os.getenv("MINIO_ACCESS_KEY", "minio-access-key"),
    aws_secret_access_key=os.getenv("MINIO_SECRET_KEY", "minio-secret-key"),
    region_name="us-east-1",
)

print("MinIO client ready.")

## Carga del dataset

Leemos el CSV raw desde el bucket Bronze.

In [ ]:
obj = client.get_object(Bucket="bronze", Key="raw/Reviews.csv")
df = pd.read_csv(io.BytesIO(obj["Body"].read()), encoding="utf-8", dtype=str)
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

## Información general y tipos de datos

Revisamos la estructura: nulos, tipos y uso de memoria.

In [ ]:
df.info()

In [ ]:
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(2)
pd.DataFrame({"nulls": null_counts, "pct": null_pct}).query("nulls > 0")

## Distribución de estrellas (Score)

La columna `Score` representa la calificación de 1 a 5 estrellas. Esta distribución nos dice si el dataset está balanceado o sesgado hacia reseñas positivas.

In [ ]:
df["Score_num"] = pd.to_numeric(df["Score"], errors="coerce")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma de estrellas
score_counts = df["Score_num"].value_counts().sort_index()
score_counts.plot(kind="bar", ax=axes[0], color=["#e74c3c", "#e67e22", "#f1c40f", "#2ecc71", "#27ae60"])
axes[0].set_title("Distribución de Estrellas")
axes[0].set_xlabel("Score (estrellas)")
axes[0].set_ylabel("Cantidad de reseñas")
axes[0].tick_params(axis="x", rotation=0)

# Porcentaje
(score_counts / len(df) * 100).round(2).plot(kind="bar", ax=axes[1], color=["#e74c3c", "#e67e22", "#f1c40f", "#2ecc71", "#27ae60"])
axes[1].set_title("Distribución de Estrellas (%)")
axes[1].set_xlabel("Score (estrellas)")
axes[1].set_ylabel("Porcentaje")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

## Distribución de urgencia

Mapeamos el Score a un nivel de urgencia (1–5):

| Urgencia | Score | Significado |
|----------|-------|-------------|
| 1 | 1 estrella | **Crítica** — Requiere atención inmediata |
| 2 | 2 estrellas | **Negativa** — Experiencia mala |
| 3 | 3 estrellas | **Neutral** — Experiencia promedio |
| 4 | 4 estrellas | **Positiva** — Buena experiencia |
| 5 | 5 estrellas | **Muy positiva** — Excelente |

Para el clasificador de triage, las clases 1 y 2 son las más importantes.

In [ ]:
def compute_urgencia(score: float) -> int:
    if pd.isna(score):
        return 0
    s = int(score)
    if s <= 1:
        return 1
    if s == 2:
        return 2
    if s == 3:
        return 3
    if s == 4:
        return 4
    return 5

df["urgencia"] = df["Score_num"].apply(compute_urgencia)

urgencia_labels = {1: "1-Crítica", 2: "2-Negativa", 3: "3-Neutral", 4: "4-Positiva", 5: "5-Muy positiva"}
urgencia_counts = df["urgencia"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 5))
urgencia_counts.plot(kind="bar", ax=ax, color=["#e74c3c", "#e67e22", "#f1c40f", "#2ecc71", "#27ae60"])
ax.set_xticklabels([urgencia_labels.get(i, str(i)) for i in urgencia_counts.index], rotation=45, ha="right")
ax.set_title("Distribución de Urgencia")
ax.set_ylabel("Cantidad de reseñas")
plt.tight_layout()
plt.show()

print(urgencia_counts)

## Análisis de longitudes de texto

La longitud del texto es importante para:
- **Chunking**: reseñas muy largas se parten en chunks para la KB
- **Embeddings**: textos más largos requieren más tokens
- **Calidad del modelo**: textos muy cortos pueden no tener suficiente señal

In [ ]:
df["review_length"] = df["Text"].fillna(""),
df["review_length"] = df["Text"].fillna("").str.len().astype(int)
df["word_count"] = df["Text"].fillna("").str.split().str.len().astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df["review_length"].clip(upper=5000).hist(bins=80, ax=axes[0], color="#3498db", edgecolor="white")
axes[0].set_title("Distribución de Longitud (caracteres, clip 5000)")
axes[0].set_xlabel("Caracteres")
axes[0].set_ylabel("Frecuencia")

df["word_count"].clip(upper=1000).hist(bins=80, ax=axes[1], color="#9b59b6", edgecolor="white")
axes[1].set_title("Distribución de Word Count (clip 1000)")
axes[1].set_xlabel("Palabras")
axes[1].set_ylabel("Frecuencia")

plt.tight_layout()
plt.show()

print(df[["review_length", "word_count"]].describe().round(1))

## Estadísticas por urgencia

Vemos cómo varía la longitud del texto según el nivel de urgencia. Las reseñas negativas suelen ser más largas y detalladas.

In [ ]:
stats = df.groupby("urgencia").agg(
    count=("Id", "count"),
    avg_length=("review_length", "mean"),
    avg_words=("word_count", "mean"),
    median_length=("review_length", "median"),
).round(1)
stats.index = [urgencia_labels.get(i, str(i)) for i in stats.index]
stats

## Ejemplos por clase de urgencia

Observamos un ejemplo de texto para cada nivel de urgencia. Esto nos ayuda a entender cualitativamente qué tipo de contenido tiene cada clase.

In [ ]:
for urg in sorted(df["urgencia"].unique()):
    if urg == 0:
        continue
    subset = df[df["urgencia"] == urg]
    example = subset.sample(1, random_state=42).iloc[0]
    label = urgencia_labels.get(urg, str(urg))
    print(f"\n{'='*60}")
    print(f"URGENCIA: {label}")
    print(f"Score: {example['Score']} | Longitud: {example['review_length']} chars | Palabras: {example['word_count']}")
    print(f"Resumen: {example['Summary']}")
    print(f"Texto: {str(example['Text'])[:300]}...")

## Resumen de hallazgos

| Aspecto | Observación |
|---------|-------------|
| **Volumen** | ~568K reseñas |
| **Nulos** | Revisar columnas con missing values |
| **Balance de clases** | Dataset sesgado hacia 5 estrellas (típico de Amazon) |
| **Longitud** | Amplia variación — desde 1 hasta miles de caracteres |
| **Implicación** | El stratified split es crucial para mantener proporciones en train/test |

### Próximos pasos

1. Ejecutar el pipeline medallion (`python -m etl_pipeline.main --step all`)
2. Generar embeddings con SentenceTransformer
3. Entrenar el clasificador de urgencia con LogisticRegression